In [ ]:
from cdhit_reader import  read_cdhit
from tqdm.notebook import tqdm
import pandas as pd
#!cdhit-est -i "bpRNA.fasta" -o "bpRNA-cdhit_0.8" -c 0.8 -T 0 -d 100 -g 1

df = pd.read_csv("/home/gkulemeyer/Documents/DiskB/Guillermo/revisitingCV/xvalRNAfolding/data/bprna/bpRNA_ACGU.csv", index_col="id")
input = "/home/gkulemeyer/Documents/DiskB/Guillermo/revisitingCV/data/bpRNA/create_splits/bpRNA-cdhit_0.8.clstr"
clusters = [c for c in read_cdhit(input)]
for c in tqdm(clusters):
    for seq in c.sequences:
        df.loc[seq.name, 'cluster'] = int(c.name.split()[-1])
        df.loc[seq.name, 'is_ref'] = seq.is_ref
        df.loc[seq.name, 'identity'] = seq.identity

In [ ]:
df.cluster.hist(bins=100)
df.shape, df.cluster.unique().shape

In [ ]:
df.groupby("cluster").count().sort_values("identity")

In [ ]:
from random import shuffle
header = True
clusters = df.cluster.unique()
for k in range(5):
    shuffle(clusters)
    test, dev, train = [], [], []
    for c in clusters:
        if len(test) <= len(df)*.1:
            test += df[df.cluster==c].index.tolist()
        elif len(dev) <= len(df)*.1:
            dev += df[df.cluster==c].index.tolist()
        else:
            train += df[df.cluster==c].index.tolist()

    print(len(test)/len(df), len(dev)/len(df), len(train)/len(df))
    # write splits in format id, fold, split
    # overwrite the file in the first fold, then append the rest
    with open(f"clustering_folds_split.csv", "w" if header else "a") as f:
        if header:
            f.write(f"id,fold_number,partition\n")
            header = False
        for i in test:
            f.write(f"{i},{k},test\n")
        for i in dev:
            f.write(f"{i},{k},valid\n")
        for i in train:
            f.write(f"{i},{k},train\n")